# Image Data Preprocessing & Augmentation Pipeline

**Dataset:** 8,091 images from `Images/`

| Stage | Description |
|---|---|
| 1 | Dataset exploration & statistics |
| 2 | Data preprocessing (resize, normalize) |
| 3 | Data augmentation (flip, rotate, color, noise) |
| 4 | Quality filtering (corrupted, too small) |
| 5 | Save augmented dataset |
| 6 | Visual report |

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'Pillow','numpy','matplotlib','tqdm','torchvision','torch','scipy'])
print('OK')

## Stage 1 — Dataset Exploration

In [ ]:
import os, json, time, shutil, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image, ImageStat
from tqdm import tqdm
from collections import Counter

IMAGES_DIR  = r"C:/Users/GIGABYTE/Downloads/ImageCaptioningproject-CNN-LSTM-main (2)/Images"
BASE_DIR    = r"C:/Users/GIGABYTE/Downloads/ImageCaptioningproject-CNN-LSTM-main (2)/ImageCaptioningproject-CNN-LSTM-main"
AUG_DIR     = os.path.join(BASE_DIR, "augmented_images")
PROC_DIR    = os.path.join(BASE_DIR, "preprocessed_images")
REPORT_JSON = os.path.join(BASE_DIR, "dataset_report.json")

os.makedirs(AUG_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

# Collect all image paths
all_paths = sorted([
    os.path.join(IMAGES_DIR, f)
    for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))
])
print(f"Found {len(all_paths):,} images")

: 

In [ ]:
# Scan all images and collect statistics
stats = []
corrupt = []

for path in tqdm(all_paths, desc="Scanning dataset"):
    try:
        img = Image.open(path)
        img.verify()          # catch corrupt files
        img = Image.open(path).convert("RGB")
        w, h = img.size
        arr  = np.array(img)
        stats.append({
            "path":      path,
            "name":      os.path.basename(path),
            "width":     w,
            "height":    h,
            "ratio":     round(w/h, 2),
            "mode":      img.mode,
            "mean_r":    float(arr[:,:,0].mean()),
            "mean_g":    float(arr[:,:,1].mean()),
            "mean_b":    float(arr[:,:,2].mean()),
            "std":       float(arr.std()),
            "size_kb":   round(os.path.getsize(path)/1024, 1),
        })
    except Exception as e:
        corrupt.append({"path": path, "error": str(e)})

print(f"Valid: {len(stats):,}  |  Corrupt: {len(corrupt)}")

widths  = [s["width"]  for s in stats]
heights = [s["height"] for s in stats]
sizes   = [s["size_kb"] for s in stats]

print(f"\nWidth  : min={min(widths)}  max={max(widths)}  mean={np.mean(widths):.0f}")
print(f"Height : min={min(heights)}  max={max(heights)}  mean={np.mean(heights):.0f}")
print(f"Size KB: min={min(sizes):.0f}  max={max(sizes):.0f}  mean={np.mean(sizes):.0f}")

In [ ]:
# Visual dashboard
fig = plt.figure(figsize=(16, 10))
fig.suptitle("Dataset Exploration Report", fontsize=14, fontweight="bold")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0,0])
ax1.hist(widths,  bins=40, color="steelblue", edgecolor="white")
ax1.set_title("Width Distribution"); ax1.set_xlabel("px")

ax2 = fig.add_subplot(gs[0,1])
ax2.hist(heights, bins=40, color="coral", edgecolor="white")
ax2.set_title("Height Distribution"); ax2.set_xlabel("px")

ax3 = fig.add_subplot(gs[0,2])
ax3.hist(sizes,   bins=40, color="mediumseagreen", edgecolor="white")
ax3.set_title("File Size (KB)"); ax3.set_xlabel("KB")

ax4 = fig.add_subplot(gs[1,0])
mean_r = np.mean([s["mean_r"] for s in stats])
mean_g = np.mean([s["mean_g"] for s in stats])
mean_b = np.mean([s["mean_b"] for s in stats])
ax4.bar(["R","G","B"],[mean_r,mean_g,mean_b], color=["red","green","blue"])
ax4.set_title("Mean Channel Values"); ax4.set_ylim(0,255)

ax5 = fig.add_subplot(gs[1,1])
ratios = [s["ratio"] for s in stats]
ax5.hist(ratios, bins=40, color="mediumpurple", edgecolor="white")
ax5.set_title("Aspect Ratio (W/H)"); ax5.set_xlabel("ratio")

ax6 = fig.add_subplot(gs[1,2])
stds = [s["std"] for s in stats]
ax6.hist(stds, bins=40, color="orange", edgecolor="white")
ax6.set_title("Pixel Std Dev (contrast)"); ax6.set_xlabel("std")

plt.savefig(os.path.join(BASE_DIR, "dataset_report.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Report saved")

## Stage 2 — Preprocessing

Standardises every image to a fixed size with normalised pixel values.

In [ ]:
# Preprocessing config
TARGET_SIZE  = (224, 224)    # standard CLIP / CNN input size
MIN_SIZE     = 64            # skip images smaller than this (px)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

# Filter out images that are too small
valid_paths = [
    s["path"] for s in stats
    if s["width"] >= MIN_SIZE and s["height"] >= MIN_SIZE
]
skipped = len(stats) - len(valid_paths)
print(f"Valid for processing: {len(valid_paths):,}  |  Skipped (too small): {skipped}")


def preprocess_image(path: str, target_size=TARGET_SIZE) -> np.ndarray:
    """
    Preprocess a single image:
    1. Load & convert to RGB
    2. Resize to target_size (aspect-preserving crop from center)
    3. Normalize with ImageNet mean/std
    Returns float32 numpy array [H, W, 3] in [0,1] range.
    """
    img = Image.open(path).convert("RGB")
    # Center crop to square first, then resize
    w, h = img.size
    s    = min(w, h)
    left = (w - s) // 2
    top  = (h - s) // 2
    img  = img.crop((left, top, left+s, top+s))
    img  = img.resize(target_size, Image.LANCZOS)
    arr  = np.array(img, dtype=np.float32) / 255.0
    arr  = (arr - IMAGENET_MEAN) / IMAGENET_STD
    return arr


# Demo: preprocess first 6 images and show side-by-side
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle("Preprocessing: Original vs Preprocessed (224×224)", fontsize=12)

for i, path in enumerate(valid_paths[:6]):
    orig = Image.open(path).convert("RGB")
    proc = preprocess_image(path)
    # De-normalise for display
    disp = np.clip(proc * IMAGENET_STD + IMAGENET_MEAN, 0, 1)

    axes[0, i].imshow(orig);  axes[0, i].axis("off")
    axes[0, i].set_title("Original", fontsize=7)
    axes[1, i].imshow(disp);  axes[1, i].axis("off")
    axes[1, i].set_title(f"224×224", fontsize=7)

plt.tight_layout(); plt.show()

## Stage 3 — Data Augmentation

Applies 8 augmentation techniques to artificially multiply the dataset size.

In [ ]:
import random
from PIL import ImageEnhance, ImageFilter, ImageOps

# ── Augmentation functions ────────────────────────────────────────────────────

def aug_horizontal_flip(img: Image.Image) -> Image.Image:
    return ImageOps.mirror(img)

def aug_vertical_flip(img: Image.Image) -> Image.Image:
    return ImageOps.flip(img)

def aug_rotate(img: Image.Image, max_angle=30) -> Image.Image:
    angle = random.uniform(-max_angle, max_angle)
    return img.rotate(angle, expand=False, fillcolor=(128,128,128))

def aug_brightness(img: Image.Image, lo=0.6, hi=1.5) -> Image.Image:
    return ImageEnhance.Brightness(img).enhance(random.uniform(lo, hi))

def aug_contrast(img: Image.Image, lo=0.6, hi=1.5) -> Image.Image:
    return ImageEnhance.Contrast(img).enhance(random.uniform(lo, hi))

def aug_saturation(img: Image.Image, lo=0.4, hi=2.0) -> Image.Image:
    return ImageEnhance.Color(img).enhance(random.uniform(lo, hi))

def aug_blur(img: Image.Image, radius=1.5) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.5, radius)))

def aug_gaussian_noise(img: Image.Image, sigma=15) -> Image.Image:
    arr   = np.array(img, dtype=np.int16)
    noise = np.random.normal(0, random.uniform(5, sigma), arr.shape).astype(np.int16)
    return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

def aug_crop_and_pad(img: Image.Image, crop_pct=0.1) -> Image.Image:
    w, h = img.size
    cx   = int(w * random.uniform(0, crop_pct))
    cy   = int(h * random.uniform(0, crop_pct))
    img  = img.crop((cx, cy, w-cx, h-cy))
    return img.resize((w, h), Image.LANCZOS)


AUGMENTATIONS = {
    "hflip":       aug_horizontal_flip,
    "vflip":       aug_vertical_flip,
    "rotate":      aug_rotate,
    "brightness":  aug_brightness,
    "contrast":    aug_contrast,
    "saturation":  aug_saturation,
    "blur":        aug_blur,
    "noise":       aug_gaussian_noise,
    "crop_pad":    aug_crop_and_pad,
}

print(f"{len(AUGMENTATIONS)} augmentation techniques loaded")

In [ ]:
# Demo: show all augmentations on one image
demo_img = Image.open(valid_paths[0]).convert("RGB").resize((224,224), Image.LANCZOS)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle("All Augmentation Techniques", fontsize=13, fontweight="bold")

axes[0,0].imshow(demo_img); axes[0,0].set_title("ORIGINAL", fontweight="bold")
axes[0,0].axis("off")

for idx, (name, fn) in enumerate(AUGMENTATIONS.items()):
    row, col = divmod(idx+1, 5)
    aug = fn(demo_img)
    axes[row, col].imshow(aug)
    axes[row, col].set_title(name, fontsize=9)
    axes[row, col].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "augmentation_demo.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Augmentation demo saved")

## Stage 4 — Apply Pipeline to Full Dataset

In [ ]:
# Config: how many augmented versions per original image
N_AUG_PER_IMAGE = 3     # 3 random augmentations per image -> 3x dataset size
PROCESS_MAX     = len(valid_paths)  # set lower (e.g. 500) to test first
SAVE_PROC       = True   # save preprocessed originals
SAVE_AUG        = True   # save augmented images

aug_manifest = []  # records what was created
errors_aug   = []

aug_names = list(AUGMENTATIONS.keys())

for path in tqdm(valid_paths[:PROCESS_MAX], desc="Preprocessing & Augmenting"):
    try:
        stem = os.path.splitext(os.path.basename(path))[0]
        img  = Image.open(path).convert("RGB")

        # ── Preprocessing: center-crop + resize to 224×224 ─────────────────
        w, h = img.size
        s    = min(w, h)
        img_sq = img.crop(((w-s)//2, (h-s)//2, (w-s)//2+s, (h-s)//2+s))
        img_proc = img_sq.resize((224, 224), Image.LANCZOS)

        if SAVE_PROC:
            proc_path = os.path.join(PROC_DIR, f"{stem}_proc.jpg")
            img_proc.save(proc_path, quality=92)

        aug_manifest.append({"original": path, "type": "preprocessed",
                             "saved_as": proc_path if SAVE_PROC else None})

        # ── Augmentation: pick N random techniques ──────────────────────────
        chosen_augs = random.sample(aug_names, min(N_AUG_PER_IMAGE, len(aug_names)))
        for aug_name in chosen_augs:
            aug_img = AUGMENTATIONS[aug_name](img_proc)
            if SAVE_AUG:
                aug_path = os.path.join(AUG_DIR, f"{stem}_{aug_name}.jpg")
                aug_img.save(aug_path, quality=88)
            aug_manifest.append({"original": path, "type": aug_name,
                                 "saved_as": aug_path if SAVE_AUG else None})

    except Exception as e:
        errors_aug.append({"path": path, "error": str(e)})

total_generated = sum(1 for m in aug_manifest if m["type"] != "preprocessed")
print(f"\nOriginals preprocessed : {PROCESS_MAX:,}")
print(f"Augmented images saved  : {total_generated:,}")
print(f"Total dataset size      : {PROCESS_MAX + total_generated:,}")
print(f"Errors                  : {len(errors_aug)}")

# Save manifest
with open(os.path.join(BASE_DIR, "augmentation_manifest.json"), "w") as f:
    json.dump(aug_manifest, f, indent=2)
print("Manifest saved")

## Stage 5 — Final Report

In [ ]:
aug_counts = Counter(m["type"] for m in aug_manifest)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Dataset Augmentation Report", fontsize=13, fontweight="bold")

# Bar chart of augmentation type counts
labels = list(aug_counts.keys())
counts = list(aug_counts.values())
colors = plt.cm.Set2(np.linspace(0, 1, len(labels)))
axes[0].barh(labels, counts, color=colors)
axes[0].set_title("Count per Augmentation Type")
axes[0].set_xlabel("Images")

# Pie chart: original vs augmented
orig_count = aug_counts.get("preprocessed", 0)
aug_total  = sum(v for k,v in aug_counts.items() if k != "preprocessed")
axes[1].pie([orig_count, aug_total],
            labels=[f"Original ({orig_count:,})", f"Augmented ({aug_total:,})"],
            autopct="%1.1f%%", colors=["steelblue","coral"], startangle=90)
axes[1].set_title("Dataset Composition")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "augmentation_report.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\n=== Pipeline Summary ===")
print(f"  Raw images          : {len(all_paths):,}")
print(f"  Corrupt/skipped     : {len(corrupt) + skipped}")
print(f"  Valid & preprocessed: {orig_count:,}")
print(f"  Augmented variants  : {aug_total:,}")
print(f"  TOTAL DATASET SIZE  : {orig_count + aug_total:,}")
print(f"  Augmentation factor : {(orig_count + aug_total) / max(orig_count,1):.1f}x")

In [ ]:
# Show a grid of augmented versions of 3 sample images
sample_stems = [os.path.splitext(os.path.basename(p))[0] for p in valid_paths[:3]]

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
fig.suptitle("Sample: Original + 3 Augmented Versions", fontsize=12, fontweight="bold")

for row, stem in enumerate(sample_stems):
    orig_proc = os.path.join(PROC_DIR, f"{stem}_proc.jpg")
    axes[row,0].imshow(Image.open(orig_proc))
    axes[row,0].set_title("preprocessed", fontsize=8, fontweight="bold")
    axes[row,0].axis("off")

    aug_files = [f for f in os.listdir(AUG_DIR) if f.startswith(stem)]
    for col, af in enumerate(aug_files[:3], start=1):
        aug_name = af.replace(stem+"_","").replace(".jpg","")
        axes[row,col].imshow(Image.open(os.path.join(AUG_DIR, af)))
        axes[row,col].set_title(aug_name, fontsize=8)
        axes[row,col].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "augmentation_grid.png"), dpi=120, bbox_inches="tight")
plt.show()